# WavLM Utterance-Level Extraction
## Extract features FOR EACH UTTERANCE using label timestamps

**Key**: Uses actual utterance boundaries (start, end) from labels, NOT fixed windows

Steps:
1. Load utterance labels from Google Drive
2. For each utterance, extract WavLM features for [start, end] segment
3. Save as NPZ

In [ ]:
# Install dependencies
!pip install -q transformers torch librosa soundfile tqdm

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted')

In [ ]:
import os
import json
import gzip
import numpy as np
import torch
from transformers import WavLMModel, Wav2Vec2Processor
import librosa
from tqdm import tqdm

# Paths
AUDIO_DIR = '/content/gdrive/MyDrive/chuckle_audio_all'
LABELS_FILE = '/content/gdrive/MyDrive/utterances_clean.jsonl'
OUTPUT_PATH = '/content/gdrive/MyDrive/wavlm_utterance_level.npz'

# Load labels
if LABELS_FILE.endswith('.gz'):
    labels = []
    with gzip.open(LABELS_FILE, 'rt') as f:
        for line in f:
            labels.append(json.loads(line))
else:
    with open(LABELS_FILE, 'r') as f:
        labels = [json.loads(line) for line in f]

print(f'Loaded {len(labels)} utterances')

# Group by video
labels_by_video = {}
for l in labels:
    vid = l['video_id'].split('.')[0]
    if vid not in labels_by_video:
        labels_by_video[vid] = []
    labels_by_video[vid].append(l)

print(f'Videos with labels: {len(labels_by_video)}')

In [ ]:
# Find audio files in all subfolders
def find_audio_files(base_dir):
    audio_files = {}
    for root, dirs, files in os.walk(base_dir):
        for f in files:
            if f.endswith(('.mp3', '.wav', '.m4a', '.flac')):
                vid = f.rsplit('.', 1)[0]
                audio_files[vid] = os.path.join(root, f)
    return audio_files

audio_files = find_audio_files(AUDIO_DIR)
print(f'Audio files found: {len(audio_files)}')

# Check overlap
vids_with_both = set(labels_by_video.keys()) & set(audio_files.keys())
print(f'Videos with BOTH audio and labels: {len(vids_with_both)}')

In [ ]:
# Load WavLM
model_name = 'microsoft/wavlm-base'
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = WavLMModel.from_pretrained(model_name)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
SR = 16000
print(f'Model loaded, using {device}')

In [ ]:
def extract_segment_features(audio_path, start_sec, end_sec, sr=16000):
    """Extract WavLM features for a specific segment [start_sec, end_sec]"""
    try:
        # Load just this segment
        y, _ = librosa.load(audio_path, offset=start_sec, duration=end_sec-start_sec, sr=sr)
        
        # Convert to float32
        y = y.astype(np.float32)
        
        # Process with WavLM
        inputs = processor(y, sampling_rate=sr, return_tensors='pt', padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items() if k != 'attention_mask'}
        
        with torch.no_grad():
            outputs = model(**inputs)
            # Mean pooling over time
            embedding = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
        
        return embedding
    except Exception as e:
        return None

In [ ]:
# Extract for all utterances
embeddings = []
labels_out = []
uids_out = []
errors = 0
skipped = 0

vids = list(vids_with_both)
print(f'Processing {len(vids)} videos...')

for vid in tqdm(vids):
    audio_path = audio_files[vid]
    
    for utt in labels_by_video[vid]:
        start = utt['start']
        end = utt['end']
        label = utt['label']
        uid = f"{vid}_{start}"
        
        # Skip very short segments (< 100ms)
        if end - start < 0.1:
            skipped += 1
            continue
        
        emb = extract_segment_features(audio_path, start, end)
        
        if emb is not None:
            embeddings.append(emb)
            labels_out.append(label)
            uids_out.append(uid)
        else:
            errors += 1

print(f'\nDone! Extracted: {len(embeddings)}, Errors: {errors}, Skipped: {skipped}')
print(f'Positive: {sum(labels_out)} ({sum(labels_out)/len(labels_out)*100:.1f}%)')

In [ ]:
# Save as NPZ
embeddings = np.array(embeddings, dtype=np.float32)
labels_out = np.array(labels_out, dtype=np.int64)

np.savez_compressed(OUTPUT_PATH,
    embeddings=embeddings,
    labels=labels_out,
    uids=np.array(uids_out))

print(f'Saved to {OUTPUT_PATH}')
print(f'Shape: embeddings={embeddings.shape}, labels={labels_out.shape}')
print(f'Positive rate: {sum(labels_out)/len(labels_out)*100:.1f}%')